# Homepage false alarms: 2x2 ablation

Notebook 03 showed that the selected model (character TF-IDF with
SGDClassifier) raised false alarms on 7.62% of legitimate homepage-like
validation URLs, versus 0.96% for other legitimate URLs. In the PhreshPhish
sample, 44.5% of phishing URLs are homepage-like, versus 7.0% of benign URLs.

This notebook tests two changes, separately and together, on identical splits:

| Arm | Training data | Representation |
|---|---|---|
| A0 baseline | Original rows | One TF-IDF over the full model text |
| A1 augmented | Original rows plus weakly labeled benign homepages | One TF-IDF |
| A2 split | Original rows | Separate TF-IDF for hostname and for the rest |
| A3 split + augmented | Original rows plus weakly labeled benign homepages | Separate TF-IDF |

Hypotheses (not established causes):

- H1: Benign homepages are rare in the training data, so the model may treat
  a missing path as evidence of phishing. Adding benign homepage examples may
  reduce that association.
- H2: With one vector, a bare domain produces a short, unusual feature vector.
  Normalising hostname and path separately may let hostname n-grams carry the
  decision for bare domains.

Data comes from `scripts/prepare_phreshphish.py`, pinned to a fixed dataset
revision. The final test file is loaded with its URL column only, for overlap
checks. Its labels are not read in this notebook.

In [1]:
import json
import sys
from pathlib import Path
from urllib.parse import urlsplit

import joblib
import numpy as np
import pandas as pd
import tldextract
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Same preprocessing function as the backend
from backend.features import url_to_model_text

DATA_DIR = ROOT / "data" / "phreshphish"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

LABELS = {"phish": 0, "benign": 1}

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

print(json.dumps(json.loads((DATA_DIR / "manifest.json").read_text()), indent=2))

{
  "dataset": "phreshphish/phreshphish",
  "revision": "eabec4b7a66324b79cc8a0ad856d1731dc26fe1a",
  "stream_seed": 42,
  "stream_buffer_size": 10000,
  "final_test_seed": 2026,
  "full_test_split_rows": 168060,
  "test_rows_removed_for_text_overlap": 10254,
  "test_rows_removed_as_duplicates": 1744,
  "final_test_pool_rows": 156062,
  "files": {
    "train_sample.parquet": {
      "rows": 30000,
      "sha256": "dc6b3fa08e3497ad1c54a2f99907f6a2aacbb4d1d2dd15deec8b4c00f0728691"
    },
    "dev_benchmark.parquet": {
      "rows": 10000,
      "sha256": "820c3fb6cce19ec28b9eefcaaf52a0f6c349e62017397ef2157790557be1fbd1"
    },
    "final_test.parquet": {
      "rows": 10000,
      "sha256": "e2587f9774924bdf7982c7f34cd1b5d9caab0bdd8f5ea86d2c8219f63ac853ff"
    }
  }
}


## Helper functions

`split_model_text` splits the backend model text into hostname and the rest
(path, query and fragment). Joining the two parts gives back the original
model text, which is asserted below. The homepage-like definition is the
same as in notebook 03.

In [2]:
def split_model_text(text):
    for position, character in enumerate(text):
        if character in "/?#":
            return text[:position], text[position:]
    return text, ""


# Bundled public suffix list snapshot, so results do not depend on the network
extract_domain = tldextract.TLDExtract(suffix_list_urls=())


def registered_domain(hostname):
    return extract_domain(hostname).top_domain_under_public_suffix or hostname


def is_homepage_like(url):
    url = url.strip()
    parsed = urlsplit(url if "://" in url else "https://" + url)
    return len(parsed.path) <= 1 and len(parsed.query) == 0


def prepare(frame):
    prepared = pd.DataFrame(index=frame.index)
    prepared["url"] = frame["url"]
    prepared["text"] = frame["url"].map(url_to_model_text)

    parts = prepared["text"].map(split_model_text)
    prepared["host"] = parts.str[0]
    prepared["rest"] = parts.str[1]
    prepared["domain"] = prepared["host"].map(registered_domain)
    prepared["homepage_like"] = frame["url"].map(is_homepage_like)

    if "label" in frame:
        prepared["y"] = frame["label"].map(LABELS)
        assert prepared["y"].notna().all(), "Unexpected labels"

    assert (prepared["host"] + prepared["rest"]).eq(prepared["text"]).all()
    return prepared

## 1. Can the notebook 03 data be reconstructed exactly?

Notebook 03 did not save its sampled URLs, so identity can only be checked
against what its saved outputs recorded. The values below are copied from
those outputs.

In [3]:
train_raw = pd.read_parquet(DATA_DIR / "train_sample.parquet")
dev_raw = pd.read_parquet(DATA_DIR / "dev_benchmark.parquet")

nb03_head = [
    "https://en.wikipedia.org/wiki/Sangiovese",
    "https://www.xvideos.com/models/thick634",
    "www.boursedirect.fr/fr/marche/new-york",
    "https://gouchevlaw.com/3-must-documents-llc/",
    "https://www.almanac.com/plant/crocuses",
]
nb03_label_counts = {"benign": 16662, "phish": 13338}

# Row index and URL of the validation false alarms listed in notebook 03
nb03_false_alarms = {
    15445: "https://shop.webtoon.com/",
    20786: "https://www.stmaryhudson.cc/",
    5064: "https://eforms.cso.ie/",
    14541: "https://www.paixdieubeer.be/",
    6378: "www.tf1info.fr/",
    23129: "https://lunchbox.io/",
    24933: "https://poczta.domena.pl/",
    10943: "https://champagne.pages.dev/",
    19383: "https://www.teamvaal.co.za/",
    5277: "https://login.kramp.com/",
    4085: "https://press.hulu.com/",
    13004: "https://business.amazon.fr/",
    8919: "https://riesa.de/",
    16756: "https://whatbeatsrock.io/",
    14182: "https://boxofficega.me/",
    23563: "https://client.shuttleid.uk/",
    27884: "https://blendos.co/",
}

nb03_validation = {
    "accuracy": 0.9376666666666666,
    "precision": 0.9803182579564489,
    "recall": 0.8774362818590704,
    "f1": 0.9260284810126582,
}
nb03_test = {
    "accuracy": 0.9004,
    "precision": 0.9794940079893475,
    "recall": 0.8000870132695236,
    "f1": 0.8807471264367817,
}


def classification_metrics(y_true, predictions):
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, pos_label=0, zero_division=0),
        "recall": recall_score(y_true, predictions, pos_label=0, zero_division=0),
        "f1": f1_score(y_true, predictions, pos_label=0, zero_division=0),
    }


# Retrain the notebook 03 pipeline on the notebook 03 random split
texts = train_raw["url"].map(url_to_model_text)
y_all = train_raw["label"].map(LABELS)
texts_train, texts_val, y_train_nb03, y_val_nb03 = train_test_split(
    texts, y_all, test_size=0.20, random_state=42, stratify=y_all,
)

nb03_model = make_pipeline(
    TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=3,
                    max_features=100_000, lowercase=True),
    SGDClassifier(loss="log_loss", random_state=42, max_iter=1000),
)
nb03_model.fit(texts_train, y_train_nb03)
retrained_validation = classification_metrics(y_val_nb03, nb03_model.predict(texts_val))

# Score the regenerated dev benchmark with the saved baseline model
saved_model = joblib.load(ROOT / "models" / "phresh_text_model.joblib")
saved_phish_column = list(saved_model.classes_).index(0)

dev_scores_saved = saved_model.predict_proba(
    dev_raw["url"].map(url_to_model_text)
)[:, saved_phish_column]
dev_metrics_saved = classification_metrics(
    dev_raw["label"].map(LABELS), np.where(dev_scores_saved >= 0.5, 0, 1)
)

retrained_scores = nb03_model.predict_proba(texts_val)[:, 0]
saved_scores = saved_model.predict_proba(texts_val)[:, saved_phish_column]

checks = {
    "Train sample: label counts match": train_raw["label"].value_counts().to_dict() == nb03_label_counts,
    "Train sample: first 5 URLs match": train_raw["url"].head(5).tolist() == nb03_head,
    "Train sample: 17 listed false alarm rows match by index and URL": all(
        train_raw.loc[index, "url"] == url for index, url in nb03_false_alarms.items()
    ),
    "Train sample: retrained validation metrics match exactly": retrained_validation == nb03_validation,
    "Train sample: retrained model scores equal saved model scores": bool(
        np.array_equal(retrained_scores, saved_scores)
    ),
    "Dev benchmark: saved model metrics match the recorded test metrics exactly": dev_metrics_saved == nb03_test,
}

display(pd.Series(checks, name="Passed").to_frame())
print("Dev benchmark metrics with saved model:", dev_metrics_saved)
print("Dev benchmark label counts:", dev_raw["label"].value_counts().to_dict())

,Passed
Train sample: label counts match,True
Train sample: first 5 URLs match,True
Train sample: 17 listed false alarm rows match by index and URL,True
Train sample: retrained validation metrics match exactly,True
Train sample: retrained model scores equal saved model scores,True
Dev benchmark: saved model metrics match the recorded test metrics exactly,True


Dev benchmark metrics with saved model: {'accuracy': 0.9004, 'precision': 0.9794940079893475, 'recall': 0.8000870132695236, 'f1': 0.8807471264367817}
Dev benchmark label counts: {'benign': 5403, 'phish': 4597}


**Interpretation.** The training sample can be checked row by row at 22
positions (5 head rows and 17 indexed false alarms) and by an identical
retrained model. The dev benchmark can only be checked through its recorded
metrics, because notebook 03 never displayed its rows. Four metrics matching
to full floating-point precision on 10,000 rows is strong evidence, but it is
not a row-level proof. The conclusion printed below follows from the checks
above.

In [4]:
if all(checks.values()):
    print(
        "All checks passed. The training sample is reconstructed exactly. "
        "The dev benchmark matches every recorded statistic of the original "
        "10k test sample, so it is treated as the same benchmark, with the "
        "caveat that no row-level record of the original exists."
    )
else:
    print(
        "At least one check failed. The regenerated data must NOT be described "
        "as the notebook 03 sample or benchmark."
    )

All checks passed. The training sample is reconstructed exactly. The dev benchmark matches every recorded statistic of the original 10k test sample, so it is treated as the same benchmark, with the caveat that no row-level record of the original exists.


## 2. Overlap checks

Overlap is counted at four levels: exact URL, model text (after removing the
scheme and a leading `www.`), hostname and registered domain. The final test
file is loaded with its URL column only.

`scripts/prepare_phreshphish.py` removed final test rows whose model text
occurs in the train sample or the dev benchmark. Hostname and registered
domain overlap were deliberately not removed, because removing them would
change the test distribution. They are reported here as a limitation.

In [5]:
train = prepare(train_raw)
dev = prepare(dev_raw)
final_urls = prepare(pd.read_parquet(DATA_DIR / "final_test.parquet", columns=["url"]))


def overlap_table(pairs):
    rows = []
    for name, reference, target in pairs:
        row = {"Pair": name, "Target rows": len(target)}
        for column in ["url", "text", "host", "domain"]:
            shared = target[column].isin(set(reference[column]))
            row[f"{column} rows"] = int(shared.sum())
            row[f"{column} %"] = round(100 * shared.mean(), 2)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Pair")


overlap = overlap_table([
    ("train sample -> dev benchmark", train, dev),
    ("train sample -> final test", train, final_urls),
    ("dev benchmark -> final test", dev, final_urls),
])
display(overlap)
overlap.to_csv(RESULTS_DIR / "04_overlap_between_sets.csv")

,Target rows,url rows,url %,text rows,text %,host rows,host %,domain rows,domain %
Pair,,,,,,,,,
train sample -> dev benchmark,10000,0,0.0,0,0.0,2139,21.39,3588,35.88
train sample -> final test,10000,0,0.0,0,0.0,2213,22.13,3630,36.30
dev benchmark -> final test,10000,0,0.0,0,0.0,2302,23.02,4691,46.91


## 3. Hostname-grouped train and validation split

All rows with the same hostname stay on the same side of the split, so
validation measures performance on unseen hostnames. `StratifiedGroupKFold`
keeps the class balance close to the full sample. The first of five folds
is used, which gives roughly 80% training and 20% validation.

For comparison, the table also shows how many validation rows in the old
random split had a hostname that also appeared in training.

In [6]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fit_index, val_index = next(splitter.split(train, train["y"], groups=train["host"]))

fit_rows = train.iloc[fit_index].copy()
val = train.iloc[val_index].copy()

old_val_hosts = train.loc[y_val_nb03.index, "host"]
old_train_hosts = set(train.loc[y_train_nb03.index, "host"])
old_val_domains = train.loc[y_val_nb03.index, "domain"]
old_train_domains = set(train.loc[y_train_nb03.index, "domain"])

split_summary = pd.DataFrame({
    "Old random split (notebook 03)": {
        "Training rows": len(y_train_nb03),
        "Validation rows": len(y_val_nb03),
        "Validation phishing share %": round(100 * (y_val_nb03 == 0).mean(), 2),
        "Validation rows with hostname seen in training %": round(100 * old_val_hosts.isin(old_train_hosts).mean(), 2),
        "Validation rows with registered domain seen in training %": round(100 * old_val_domains.isin(old_train_domains).mean(), 2),
    },
    "Hostname-grouped split (this notebook)": {
        "Training rows": len(fit_rows),
        "Validation rows": len(val),
        "Validation phishing share %": round(100 * (val["y"] == 0).mean(), 2),
        "Validation rows with hostname seen in training %": round(100 * val["host"].isin(set(fit_rows["host"])).mean(), 2),
        "Validation rows with registered domain seen in training %": round(100 * val["domain"].isin(set(fit_rows["domain"])).mean(), 2),
    },
})
display(split_summary)

slice_counts = pd.crosstab(
    val["y"].map({0: "phish", 1: "benign"}),
    val["homepage_like"].map({True: "homepage-like", False: "other"}),
    margins=True,
)
print("Validation slice counts:")
display(slice_counts)

dev_slice_counts = pd.crosstab(
    dev["y"].map({0: "phish", 1: "benign"}),
    dev["homepage_like"].map({True: "homepage-like", False: "other"}),
    margins=True,
)
print("Dev benchmark slice counts:")
display(dev_slice_counts)

,Old random split (notebook 03),Hostname-grouped split (this notebook)
Training rows,24000.00,24000.00
Validation rows,6000.00,6000.00
Validation phishing share %,44.47,44.47
Validation rows with hostname seen in training %,31.30,0.00
Validation rows with registered domain seen in training %,58.20,42.08


Validation slice counts:


homepage_like,homepage-like,other,All
y,,,
benign,229,3103,3332
phish,1227,1441,2668
All,1456,4544,6000


Dev benchmark slice counts:


homepage_like,homepage-like,other,All
y,,,
benign,455,4948,5403
phish,2122,2475,4597
All,2577,7423,10000


## 4. Predefined success criteria

These criteria were written before any arm was trained and are not changed
afterwards. Each arm is compared with A0, which is retrained on the same
hostname-grouped training fold. The saved notebook 03 model is not used as
A0 here, because it was trained on rows that are now in the validation fold.

Measured on the hostname-grouped validation fold:

- **C1 (primary):** benign homepage false alarm rate at threshold 0.5 is at
  most half of A0.
- **C2 (no hidden recall cost):** at the threshold that gives each arm the
  same overall benign false alarm rate as A0 at 0.5 (chosen on validation),
  overall phishing recall is at least A0 minus 0.01. A0 is also evaluated at
  its own matched threshold, which gives the same false alarm rate.
- **C3 (phishing homepages):** homepage-like phishing recall at threshold 0.5
  falls by no more than 0.03 compared with A0.
- **C4 (threshold independent):** phishing PR-AUC (average precision) is not
  lower than A0.

Measured on the dev benchmark:

- **C5 (confirmation):** benign homepage false alarm rate at threshold 0.5 is
  lower than A0.

An arm passes only if C1 to C5 all hold. Point estimates decide pass or fail.
Confidence intervals are reported next to them, and an improvement whose
interval includes zero is reported as uncertain.

Uncertainty: Wilson 95% intervals for each rate, and a 95% bootstrap interval
for each difference from A0. The bootstrap resamples hostnames with
replacement (1,000 repetitions), because rows from the same hostname are not
independent.

## 5. Weakly labeled benign homepage augmentation

Rules, fixed before training:

1. Only the training fold is used. Validation and dev benchmark labels are
   never consulted.
2. A hostname is a candidate only if **all** its training fold rows are
   benign. Hostnames with conflicting labels are excluded.
3. Hostnames that already have a homepage-like original row are skipped,
   because a real observation already exists.
4. Each remaining hostname gets one generated example: the bare hostname as
   model text, with an empty path, labeled benign.
5. Generated rows are **weak labels**. A legitimate deep link suggests, but
   does not prove, that its homepage is legitimate. They get sample weight
   0.5, while original rows get 1.0.
6. Phishing hostnames are not augmented.

The generated rows also take part in fitting the TF-IDF vocabulary and IDF
weights. TF-IDF fitting does not support sample weights, so in that step they
count as ordinary documents.

In [7]:
AUGMENT_WEIGHT = 0.5

host_labels = fit_rows.groupby("host")["y"].agg(["min", "max"])
host_has_homepage = fit_rows.groupby("host")["homepage_like"].any()

benign_only = host_labels.index[(host_labels["min"] == 1) & (host_labels["max"] == 1)]
phish_only = host_labels.index[(host_labels["min"] == 0) & (host_labels["max"] == 0)]
conflicting = host_labels.index[host_labels["min"] != host_labels["max"]]

already_has_homepage = [host for host in benign_only if host_has_homepage[host]]
empty_host = [host for host in benign_only if host == ""]
generated_hosts = [
    host for host in benign_only
    if host != "" and not host_has_homepage[host]
]

augmented = pd.DataFrame({
    "text": generated_hosts,
    "host": generated_hosts,
    "rest": "",
    "y": 1,
})

augmentation_counts = {
    "Training fold rows": len(fit_rows),
    "Training fold hostnames": len(host_labels),
    "Benign-only hostnames": len(benign_only),
    "Phishing-only hostnames": len(phish_only),
    "Conflicting hostnames (excluded)": len(conflicting),
    "Benign-only hostnames with an existing homepage-like row (skipped)": len(already_has_homepage),
    "Benign-only hostnames with empty hostname (skipped)": len(empty_host),
    "Generated weakly labeled homepages": len(augmented),
    "Generated rows as % of training fold rows": round(100 * len(augmented) / len(fit_rows), 2),
    "Sample weight for generated rows": AUGMENT_WEIGHT,
    "Benign homepage-like rows in training fold before augmentation": int(
        (fit_rows["homepage_like"] & (fit_rows["y"] == 1)).sum()
    ),
}

display(pd.Series(augmentation_counts, name="Count").to_frame())
(RESULTS_DIR / "04_augmentation_counts.json").write_text(
    json.dumps(augmentation_counts, indent=2) + "\n"
)
display(augmented.head(10))

,Count
Training fold rows,24000.00
Training fold hostnames,17243.00
Benign-only hostnames,8573.00
Phishing-only hostnames,8660.00
Conflicting hostnames (excluded),10.00
Benign-only hostnames with an existing homepage-like row (skipped),932.00
Benign-only hostnames with empty hostname (skipped),0.00
Generated weakly labeled homepages,7641.00
Generated rows as % of training fold rows,31.84
Sample weight for generated rows,0.50


,text,host,rest,y
0,01.cstoreoffice.com,01.cstoreoffice.com,,1
1,0tnews.com,0tnews.com,,1
2,1.html-load.com,1.html-load.com,,1
3,100percentpure.com,100percentpure.com,,1
4,1045theteam.com,1045theteam.com,,1
5,1066online.com,1066online.com,,1
6,10tv.com,10tv.com,,1
7,123bearing.com,123bearing.com,,1
8,123bodyscan.com,123bodyscan.com,,1
9,123pneus.fr,123pneus.fr,,1


## 6. Train the four arms

Fixed settings for all arms: character n-grams 3 to 5, `min_df=3`,
lowercase, `SGDClassifier(loss="log_loss", random_state=42, max_iter=1000)`.
The single representation uses `max_features=100_000`. The split
representation uses 50,000 for the hostname and 50,000 for the rest, so the
total feature budget is the same.

In [8]:
TFIDF = dict(analyzer="char", ngram_range=(3, 5), min_df=3, lowercase=True)

ARMS = {
    "A0 baseline": ("single", False),
    "A1 augmented": ("single", True),
    "A2 split": ("split", False),
    "A3 split + augmented": ("split", True),
}


def make_model(representation):
    if representation == "single":
        features = ColumnTransformer([
            ("text", TfidfVectorizer(**TFIDF, max_features=100_000), "text"),
        ])
    else:
        features = ColumnTransformer([
            ("host", TfidfVectorizer(**TFIDF, max_features=50_000), "host"),
            ("rest", TfidfVectorizer(**TFIDF, max_features=50_000), "rest"),
        ])
    return make_pipeline(
        features,
        SGDClassifier(loss="log_loss", random_state=42, max_iter=1000),
    )


INPUT_COLUMNS = ["text", "host", "rest"]

models = {}
for arm, (representation, use_augmentation) in ARMS.items():
    training = fit_rows[INPUT_COLUMNS + ["y"]].assign(weight=1.0)
    if use_augmentation:
        training = pd.concat(
            [training, augmented.assign(weight=AUGMENT_WEIGHT)],
            ignore_index=True,
        )

    model = make_model(representation)
    model.fit(
        training[INPUT_COLUMNS],
        training["y"].astype(int),
        sgdclassifier__sample_weight=training["weight"].to_numpy(),
    )
    models[arm] = model
    print(f"{arm}: trained on {len(training)} rows")


def phishing_scores(model, frame):
    column = list(model.classes_).index(0)
    return model.predict_proba(frame[INPUT_COLUMNS])[:, column]


val_scores = {arm: phishing_scores(model, val) for arm, model in models.items()}
dev_scores = {arm: phishing_scores(model, dev) for arm, model in models.items()}

A0 baseline: trained on 24000 rows


A1 augmented: trained on 31641 rows


A2 split: trained on 24000 rows


A3 split + augmented: trained on 31641 rows


## 7. Metrics

In [9]:
def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return (np.nan, np.nan)
    rate = successes / total
    denominator = 1 + z**2 / total
    centre = (rate + z**2 / (2 * total)) / denominator
    margin = z * np.sqrt(rate * (1 - rate) / total + z**2 / (4 * total**2)) / denominator
    return (centre - margin, centre + margin)


def threshold_for_false_alarm_rate(scores, y, target_rate):
    # Lowest threshold whose benign false alarm rate does not exceed the target
    benign_scores = np.sort(scores[y == 1])[::-1]
    allowed = int(np.floor(target_rate * len(benign_scores) + 1e-9))
    return float(np.nextafter(benign_scores[allowed], np.inf))


def evaluate(frame, scores, threshold):
    y = frame["y"].to_numpy()
    home = frame["homepage_like"].to_numpy()
    predictions = np.where(scores >= threshold, 0, 1)
    flagged = predictions == 0

    benign_home = (y == 1) & home
    phish_home = (y == 0) & home
    benign_home_alarms = int((flagged & benign_home).sum())
    phish_home_detected = int((flagged & phish_home).sum())

    result = {"threshold": threshold, **classification_metrics(y, predictions)}
    result["benign false alarm rate"] = flagged[y == 1].mean()
    result["PR-AUC"] = average_precision_score(y == 0, scores)

    result["benign homepages n"] = int(benign_home.sum())
    result["benign homepage false alarms"] = benign_home_alarms
    result["benign homepage false alarm rate"] = benign_home_alarms / benign_home.sum()
    low, high = wilson_interval(benign_home_alarms, benign_home.sum())
    result["benign homepage FAR 95% CI"] = f"{low:.4f} to {high:.4f}"

    result["phishing homepages n"] = int(phish_home.sum())
    result["phishing homepages detected"] = phish_home_detected
    result["phishing homepage recall"] = phish_home_detected / phish_home.sum()
    low, high = wilson_interval(phish_home_detected, phish_home.sum())
    result["phishing homepage recall 95% CI"] = f"{low:.4f} to {high:.4f}"
    return result


baseline = "A0 baseline"
baseline_far = evaluate(val, val_scores[baseline], 0.5)["benign false alarm rate"]

matched_thresholds = {
    arm: threshold_for_false_alarm_rate(scores, val["y"].to_numpy(), baseline_far)
    for arm, scores in val_scores.items()
}

tables = {}
for name, frame, scores in [("validation", val, val_scores), ("dev benchmark", dev, dev_scores)]:
    rows = {}
    for arm in ARMS:
        rows[(arm, "0.5")] = evaluate(frame, scores[arm], 0.5)
        # The matched threshold is chosen on validation and reused on the dev benchmark
        rows[(arm, "matched")] = evaluate(frame, scores[arm], matched_thresholds[arm])
    tables[name] = pd.DataFrame(rows).T
    tables[name].index.names = ["Arm", "Threshold rule"]
    tables[name].to_csv(RESULTS_DIR / f"04_metrics_{name.replace(' ', '_')}.csv")

print(f"Target overall benign false alarm rate for matching: {baseline_far:.4f}")
print("Matched thresholds (chosen on validation):",
      {arm: round(value, 4) for arm, value in matched_thresholds.items()})

summary_columns = [
    "threshold", "accuracy", "precision", "recall", "f1",
    "benign false alarm rate", "PR-AUC",
]
homepage_columns = [
    "benign homepages n", "benign homepage false alarms",
    "benign homepage false alarm rate", "benign homepage FAR 95% CI",
    "phishing homepages n", "phishing homepages detected",
    "phishing homepage recall", "phishing homepage recall 95% CI",
]

for name, table in tables.items():
    print(f"\n=== {name}: overall ===")
    display(table[summary_columns].astype({c: float for c in summary_columns}).round(4))
    print(f"=== {name}: homepage-like slices ===")
    display(table[homepage_columns].round(4))

Target overall benign false alarm rate for matching: 0.0150
Matched thresholds (chosen on validation): {'A0 baseline': 0.499, 'A1 augmented': 0.4298, 'A2 split': 0.5561, 'A3 split + augmented': 0.4527}

=== validation: overall ===


threshold  accuracy  precision  recall      f1  benign false alarm rate  PR-AUC
Arm                  Threshold rule                                                                                 
A0 baseline          0.5                0.5000    0.9240     0.9784  0.8478  0.9084                   0.0150  0.9837
                     matched            0.4990    0.9245     0.9784  0.8490  0.9091                   0.0150  0.9837
A1 augmented         0.5                0.5000    0.8948     0.9918  0.7699  0.8668                   0.0051  0.9799
                     matched            0.4298    0.9113     0.9776  0.8193  0.8915                   0.0150  0.9799
A2 split             0.5                0.5000    0.9345     0.9695  0.8804  0.9228                   0.0222  0.9839
                     matched            0.5561    0.9257     0.9785  0.8516  0.9106                   0.0150  0.9839
A3 split + augmented 0.5                0.5000    0.9052     0.9857  0.7984  0.8822                   0.0093  0.9819
                     matched            0.4527    0.9168     0.9780  0.8317  0.8989                   0.0150  0.9819

=== validation: homepage-like slices ===


benign homepages n benign homepage false alarms benign homepage false alarm rate benign homepage FAR 95% CI phishing homepages n phishing homepages detected  \
Arm                  Threshold rule                                                                                                                                                                
A0 baseline          0.5                           229                           14                         0.061135           0.0368 to 0.1000                 1227                        1055   
                     matched                       229                           14                         0.061135           0.0368 to 0.1000                 1227                        1055   
A1 augmented         0.5                           229                            4                         0.017467           0.0068 to 0.0440                 1227                         959   
                     matched                       229                            7                         0.030568           0.0149 to 0.0617                 1227                        1007   
A2 split             0.5                           229                           26                         0.113537           0.0787 to 0.1612                 1227                        1109   
                     matched                       229                           14                         0.061135           0.0368 to 0.1000                 1227                        1071   
A3 split + augmented 0.5                           229                            3                           0.0131           0.0045 to 0.0378                 1227                         973   
                     matched                       229                            7                         0.030568           0.0149 to 0.0617                 1227                        1016   

                                    phishing homepage recall phishing homepage recall 95% CI  
Arm                  Threshold rule                                                           
A0 baseline          0.5                            0.859821                0.8393 to 0.8781  
                     matched                        0.859821                0.8393 to 0.8781  
A1 augmented         0.5                            0.781581                0.7576 to 0.8038  
                     matched                        0.820701                0.7982 to 0.8412  
A2 split             0.5                             0.90383                0.8861 to 0.9191  
                     matched                        0.872861                0.8530 to 0.8903  
A3 split + augmented 0.5                            0.792991                0.7694 to 0.8147  
                     matched                        0.828036                0.8059 to 0.8481


=== dev benchmark: overall ===


threshold  accuracy  precision  recall      f1  benign false alarm rate  PR-AUC
Arm                  Threshold rule                                                                                 
A0 baseline          0.5                0.5000    0.8979     0.9791  0.7949  0.8774                   0.0144  0.9772
                     matched            0.4990    0.8988     0.9789  0.7970  0.8787                   0.0146  0.9772
A1 augmented         0.5                0.5000    0.8310     0.9866  0.6411  0.7772                   0.0074  0.9666
                     matched            0.4298    0.8674     0.9784  0.7276  0.8346                   0.0137  0.9666
A2 split             0.5                0.5000    0.8936     0.9616  0.8005  0.8737                   0.0272  0.9705
                     matched            0.5561    0.8804     0.9767  0.7579  0.8535                   0.0154  0.9705
A3 split + augmented 0.5                0.5000    0.8503     0.9814  0.6874  0.8085                   0.0111  0.9673
                     matched            0.4527    0.8685     0.9743  0.7333  0.8368                   0.0165  0.9673

=== dev benchmark: homepage-like slices ===


benign homepages n benign homepage false alarms benign homepage false alarm rate benign homepage FAR 95% CI phishing homepages n phishing homepages detected  \
Arm                  Threshold rule                                                                                                                                                                
A0 baseline          0.5                           455                           22                         0.048352           0.0321 to 0.0721                 2122                        1765   
                     matched                       455                           23                         0.050549           0.0339 to 0.0747                 2122                        1771   
A1 augmented         0.5                           455                           11                         0.024176           0.0136 to 0.0428                 2122                        1399   
                     matched                       455                           12                         0.026374           0.0151 to 0.0455                 2122                        1504   
A2 split             0.5                           455                           61                         0.134066           0.1058 to 0.1685                 2122                        1793   
                     matched                       455                           30                         0.065934           0.0466 to 0.0926                 2122                        1734   
A3 split + augmented 0.5                           455                           10                         0.021978           0.0120 to 0.0400                 2122                        1505   
                     matched                       455                           11                         0.024176           0.0136 to 0.0428                 2122                        1589   

                                    phishing homepage recall phishing homepage recall 95% CI  
Arm                  Threshold rule                                                           
A0 baseline          0.5                            0.831762                0.8152 to 0.8471  
                     matched                         0.83459                0.8182 to 0.8498  
A1 augmented         0.5                            0.659284                0.6388 to 0.6791  
                     matched                        0.708765                0.6891 to 0.7277  
A2 split             0.5                            0.844958                0.8289 to 0.8597  
                     matched                        0.817154                0.8001 to 0.8330  
A3 split + augmented 0.5                            0.709237                0.6896 to 0.7282  
                     matched                        0.748822                0.7299 to 0.7668

### Differences from A0 with hostname bootstrap intervals

Threshold 0.5. Each repetition resamples hostnames with replacement and
recomputes every metric for every arm on the same resample, so the
differences are paired.

In [10]:
def bootstrap_differences(frame, scores_by_arm, repetitions=1000, seed=42):
    rng = np.random.default_rng(seed)
    codes, unique_hosts = pd.factorize(frame["host"])
    n_groups = len(unique_hosts)

    y = frame["y"].to_numpy()
    home = frame["homepage_like"].to_numpy()

    def group_sum(values):
        return np.bincount(codes, weights=values.astype(float), minlength=n_groups)

    benign_home_g = group_sum((y == 1) & home)
    phish_home_g = group_sum((y == 0) & home)
    phish_g = group_sum(y == 0)

    per_arm = {}
    for arm, scores in scores_by_arm.items():
        flagged = scores >= 0.5
        per_arm[arm] = {
            "alarms": group_sum(flagged & (y == 1) & home),
            "home_detected": group_sum(flagged & (y == 0) & home),
            "detected": group_sum(flagged & (y == 0)),
        }

    samples = {arm: {m: [] for m in ["benign homepage FAR", "phishing homepage recall", "recall", "PR-AUC"]}
               for arm in scores_by_arm}

    for _ in range(repetitions):
        counts = np.bincount(rng.integers(0, n_groups, n_groups), minlength=n_groups)
        row_weights = counts[codes]
        for arm, scores in scores_by_arm.items():
            parts = per_arm[arm]
            samples[arm]["benign homepage FAR"].append(counts @ parts["alarms"] / (counts @ benign_home_g))
            samples[arm]["phishing homepage recall"].append(counts @ parts["home_detected"] / (counts @ phish_home_g))
            samples[arm]["recall"].append(counts @ parts["detected"] / (counts @ phish_g))
            samples[arm]["PR-AUC"].append(average_precision_score(y == 0, scores, sample_weight=row_weights))

    rows = {}
    for arm in scores_by_arm:
        if arm == baseline:
            continue
        for metric, values in samples[arm].items():
            difference = np.array(values) - np.array(samples[baseline][metric])
            low, high = np.percentile(difference, [2.5, 97.5])
            rows[(arm, metric)] = {
                "95% CI low": low,
                "95% CI high": high,
                "interval excludes 0": bool(low > 0 or high < 0),
            }
    return pd.DataFrame(rows).T


METRIC_COLUMNS = {
    "benign homepage FAR": "benign homepage false alarm rate",
    "phishing homepage recall": "phishing homepage recall",
    "recall": "recall",
    "PR-AUC": "PR-AUC",
}

bootstrap_tables = {}
for name, frame, scores in [("validation", val, val_scores), ("dev benchmark", dev, dev_scores)]:
    point = tables[name].xs("0.5", level="Threshold rule")
    table = bootstrap_differences(frame, scores)
    table.insert(0, "difference from A0", [
        point.loc[arm, METRIC_COLUMNS[metric]] - point.loc[baseline, METRIC_COLUMNS[metric]]
        for arm, metric in table.index
    ])
    table.index.names = ["Arm", "Metric"]
    bootstrap_tables[name] = table
    table.to_csv(RESULTS_DIR / f"04_bootstrap_{name.replace(' ', '_')}.csv")
    print(f"=== {name} ===")
    display(table.astype({"difference from A0": float, "95% CI low": float, "95% CI high": float}).round(4))

=== validation ===


difference from A0  95% CI low  95% CI high interval excludes 0
Arm                  Metric                                                                                   
A1 augmented         benign homepage FAR                  -0.0437     -0.0702      -0.0179                True
                     phishing homepage recall             -0.0782     -0.0923      -0.0640                True
                     recall                               -0.0780     -0.1026      -0.0597                True
                     PR-AUC                               -0.0038     -0.0051      -0.0027                True
A2 split             benign homepage FAR                   0.0524      0.0172       0.0881                True
                     phishing homepage recall              0.0440      0.0310       0.0576                True
                     recall                                0.0326      0.0209       0.0445                True
                     PR-AUC                                0.0002     -0.0027       0.0027               False
A3 split + augmented benign homepage FAR                  -0.0480     -0.0769      -0.0219                True
                     phishing homepage recall             -0.0668     -0.0808      -0.0525                True
                     recall                               -0.0495     -0.0673      -0.0347                True
                     PR-AUC                               -0.0018     -0.0043       0.0004               False

=== dev benchmark ===


difference from A0  95% CI low  95% CI high interval excludes 0
Arm                  Metric                                                                                   
A1 augmented         benign homepage FAR                  -0.0242     -0.0379      -0.0096                True
                     phishing homepage recall             -0.1725     -0.1881      -0.1565                True
                     recall                               -0.1538     -0.1645      -0.1431                True
                     PR-AUC                               -0.0107     -0.0120      -0.0092                True
A2 split             benign homepage FAR                   0.0857      0.0591       0.1131                True
                     phishing homepage recall              0.0132     -0.0010       0.0272               False
                     recall                                0.0057     -0.0056       0.0171               False
                     PR-AUC                               -0.0068     -0.0086      -0.0050                True
A3 split + augmented benign homepage FAR                  -0.0264     -0.0415      -0.0111                True
                     phishing homepage recall             -0.1225     -0.1365      -0.1079                True
                     recall                               -0.1075     -0.1183      -0.0951                True
                     PR-AUC                               -0.0100     -0.0119      -0.0079                True

## 8. Success criteria

In [11]:
val_05 = tables["validation"].xs("0.5", level="Threshold rule")
val_matched = tables["validation"].xs("matched", level="Threshold rule")
dev_05 = tables["dev benchmark"].xs("0.5", level="Threshold rule")

criteria_rows = {}
for arm in ARMS:
    if arm == baseline:
        continue
    c1 = val_05.loc[arm, "benign homepage false alarm rate"] <= 0.5 * val_05.loc[baseline, "benign homepage false alarm rate"]
    c2 = val_matched.loc[arm, "recall"] >= val_matched.loc[baseline, "recall"] - 0.01
    c3 = val_05.loc[arm, "phishing homepage recall"] >= val_05.loc[baseline, "phishing homepage recall"] - 0.03
    c4 = val_05.loc[arm, "PR-AUC"] >= val_05.loc[baseline, "PR-AUC"]
    c5 = dev_05.loc[arm, "benign homepage false alarm rate"] < dev_05.loc[baseline, "benign homepage false alarm rate"]
    criteria_rows[arm] = {
        "C1 homepage FAR <= half of A0": bool(c1),
        "C2 recall at matched FAR >= A0 - 0.01": bool(c2),
        "C3 phishing homepage recall >= A0 - 0.03": bool(c3),
        "C4 PR-AUC >= A0": bool(c4),
        "C5 dev homepage FAR < A0": bool(c5),
        "PASS": bool(c1 and c2 and c3 and c4 and c5),
    }

criteria = pd.DataFrame(criteria_rows).T
criteria.to_csv(RESULTS_DIR / "04_criteria.csv")
display(criteria)

reference = pd.DataFrame({
    "A0 value (validation)": {
        "benign homepage FAR at 0.5": val_05.loc[baseline, "benign homepage false alarm rate"],
        "C1 limit (half)": 0.5 * val_05.loc[baseline, "benign homepage false alarm rate"],
        "recall at matched FAR": val_matched.loc[baseline, "recall"],
        "phishing homepage recall at 0.5": val_05.loc[baseline, "phishing homepage recall"],
        "PR-AUC": val_05.loc[baseline, "PR-AUC"],
    }
})
display(reference.astype(float).round(4))

,C1 homepage FAR <= half of A0,C2 recall at matched FAR >= A0 - 0.01,C3 phishing homepage recall >= A0 - 0.03,C4 PR-AUC >= A0,C5 dev homepage FAR < A0,PASS
A1 augmented,True,False,False,False,True,False
A2 split,False,True,True,True,False,False
A3 split + augmented,True,False,False,False,True,False


,A0 value (validation)
benign homepage FAR at 0.5,0.0611
C1 limit (half),0.0306
recall at matched FAR,0.8490
phishing homepage recall at 0.5,0.8598
PR-AUC,0.9837


## 9. Illustrative probe URLs

These URLs have **no verified labels**. They are shown only to make the
behaviour visible and were not used to choose between arms. Scores are
uncalibrated model outputs, not probabilities of phishing.

In [12]:
probe_urls = [
    "netflix.com", "www.netflix.com", "paypal.com", "amazon.com",
    "google.com", "github.com", "hvl.no", "vg.no",
    "netf1ix.com", "paypa1.com", "amaz0n.com", "rnicrosoft.com",
    "paypal.com.security-check.example.com",
    "http://192.168.1.100/login",
    "https://github.com/login",
]
probes = prepare(pd.DataFrame({"url": probe_urls}))
display(pd.DataFrame(
    {arm: phishing_scores(model, probes) for arm, model in models.items()},
    index=probe_urls,
).round(3))

,A0 baseline,A1 augmented,A2 split,A3 split + augmented
netflix.com,0.563,0.307,0.461,0.313
www.netflix.com,0.563,0.307,0.461,0.313
paypal.com,0.580,0.275,0.442,0.258
amazon.com,0.152,0.059,0.091,0.058
google.com,0.900,0.671,0.713,0.604
github.com,0.881,0.678,0.690,0.579
hvl.no,0.498,0.337,0.540,0.339
vg.no,0.492,0.340,0.554,0.363
netf1ix.com,0.442,0.160,0.243,0.145
paypa1.com,0.538,0.287,0.454,0.289
